## Dependencies

In [1]:
## libraries
import os
import sys
import pandas as pd
from pathlib import Path
from itertools import combinations

## path
root = Path.cwd().resolve().parent
sys.path.insert(0, str(root))

## modules
from src.data.builders import load_processed_data
from src.estimators.factories.registry import load_estimators
from src.evaluators.metrics import consensus_metrics
from src.evaluators.training import fit_predict_frontier
from src.evaluators.config import (
    FEAT_X, 
    FEAT_Z, 
    TARGET
)

## Initalization

In [2]:
## setup
data = load_processed_data()
models = load_estimators()

## Training

In [3]:
## train models
frontiers = dict()
for name, model in models.items():
    print(f"training {name}...")
    fit_result = fit_predict_frontier(
        data = data,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        estimator_c = model.estimator_c,
        estimator_r = model.estimator_r,
        target = TARGET,
        n_repeat = 30  ## number of repeats for each training cycle
    )
    frontiers[name] = fit_result["y_pred"]

## compute pairwise consensus
results_list = list()
for (i, y_true), (j, y_pred) in combinations(iterable = frontiers.items(), r = 2):
    
    ## calculate all metrics
    metrics = consensus_metrics(y_true = y_true, y_pred = y_pred)
    results_list.append({
        "model_i": i,
        "model_j": j,
        **metrics
    })

training linear_quantile...
training linear_convex...
training linear_laws...
training forest_quantile...
training boosted_quantile...
training xgboost_quantile...
training neural_quantile...
training neural_expectile...
training neural_convex...


## Post-Processing

In [4]:
## convert to dataframe
results_data = pd.DataFrame(results_list)

## Results

In [5]:
## preview results
results_data.mean(numeric_only = True)

r      0.891695
rho    0.965748
tau    0.878889
dcr    0.893792
rbo    0.833152
dtype: float64